# Modeling for Drug Intelligence

## Objective

The objective of this notebook is to build predictive models for:

* drug effectiveness
* drug demand

## Models Used

* Linear Regression (baseline)
* Random Forest
* XGBoost

## Importance

These models help transform insights into predictions that can drive business decisions such as:

* identifying high-performing drugs
* optimizing supply


In [15]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import xgboost as xgb

In [16]:
df = pd.read_csv("../data/Final_data/final_drug_dataset.csv")

df.head()

,drugName,avg_sentiment,avg_rating,review_count,avg_usefulness,demand,demand_scaled,effectiveness_score,sentiment_per_review,engagement_score,popularity,top_side_effects,sentiment_rating_proxy,sentiment_usefulness,rating_variation_proxy
0,A + D Cracked Skin Relief,0.178800,10.000000,1,6.000000,39.880000,0.008923,10.000000,0.089400,6.0,low,[],0.1788,1.072800,0.142857
1,A / B Otic,0.329100,10.000000,1,20.000000,123.910000,0.014811,10.000000,0.164550,20.0,low,"[('pain', 1)]",0.3291,6.582000,0.047619
2,Abacavir / dolutegravir / lamivudine,0.023739,7.575758,33,14.303030,203.980000,0.020421,7.575758,0.000698,472.0,high,"[('headache', 7), ('pain', 6), ('nausea', 4)]",0.7834,0.339545,2.156436
3,Abacavir / lamivudine / zidovudine,-0.077200,9.000000,1,1.000000,17.280000,0.007340,9.000000,-0.038600,1.0,low,[],-0.0772,-0.077200,0.500000
4,Abatacept,-0.111921,7.000000,14,57.642857,193.093571,0.019658,7.000000,-0.007461,807.0,high,"[('pain', 5), ('nausea', 2), ('rash', 1)]",-1.5669,-6.451471,0.238733


In [17]:
df_original = df.copy()

In [18]:
df_model = df.drop(columns=['drugName', 'top_side_effects'])

In [19]:
df_model = pd.get_dummies(df_model, columns=['popularity'], drop_first=True)

In [20]:
X = df_model.drop(columns=['effectiveness_score', 'avg_rating'])
y = df_model['effectiveness_score']

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [22]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

mse = mean_squared_error(y_test, y_pred_lr)
rmse = np.sqrt(mse)

print("LR R2:", r2_score(y_test, y_pred_lr))
print("LR RMSE:", rmse)

LR R2: 0.12674373508057601
LR RMSE: 1.9863908868926983


In [23]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)

print("RF R2:", r2_score(y_test, y_pred_rf))
print("RF RMSE:", rmse_rf)

RF R2: 0.02228916718722429
RF RMSE: 2.101837192132158


In [24]:
xgb_model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)

xgb_model.set_params(tree_method="hist")



xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

mse_xgb = mean_squared_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mse_xgb)

print("XGB R2:", r2_score(y_test, y_pred_xgb))
print("XGB RMSE:", rmse_xgb)

XGB R2: 0.006145155131390134
XGB RMSE: 2.11911896726865


In [25]:
X_d = df_model.drop(columns=['demand', 'demand_scaled'])
y_d = df_model['demand']

In [26]:
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42
)

In [27]:
xgb_demand = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6
)

xgb_demand.fit(X_train_d, y_train_d)


xgb_demand.set_params(tree_method="hist")

y_pred_d = xgb_demand.predict(X_test_d)

mse_d = mean_squared_error(y_test_d, y_pred_d)
rmse_d = np.sqrt(mse_d)

print("Demand R2:", r2_score(y_test_d, y_pred_d))
print("Demand RMSE:", rmse_d)

Demand R2: 0.9006592440066156
Demand RMSE: 224.52509111041178


In [28]:
joblib.dump(xgb_model, "../models/effectiveness_model.pkl")
joblib.dump(xgb_demand, "../models/demand_model.pkl")

joblib.dump(X.columns.tolist(), "../models/effectiveness_features.pkl")
joblib.dump(X_d.columns.tolist(), "../models/demand_features.pkl")

['../models/demand_features.pkl']